# 数据预处理

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, normalize
import warnings
warnings.filterwarnings('ignore')

# CS231n 风格的相对误差计算与自动判题函数
def rel_error(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

def check_result(name, user_ans, expected_ans):
    if user_ans is None:
        print(f"❌ [{name}] 未完成: 请填补 TODO 代码。")
        return
    try:
        err = rel_error(user_ans, expected_ans)
        if err < 1e-5:
            print(f"✅ [{name}] 测试通过! (相对误差: {err:.4e})")
        else:
            print(f"❌ [{name}] 测试失败! 相对误差过大: {err:.4e}")
            print(f"   你的结果前几项: {np.array(user_ans)[:3]}")
            print(f"   预期结果前几项: {np.array(expected_ans)[:3]}")
    except Exception as e:
        print(f"❌ [{name}] 判题时发生代码异常: {e}")

print("✅ 环境初始化与判题函数加载完成！")

✅ 环境初始化与判题函数加载完成！


In [21]:
# 构建一份含有各种瑕疵的模拟数据
np.random.seed(42)
data = {
    'Price': [100, 150, np.nan, 200, 250, 150, 300, 3000, 180, 210], # 包含缺失值(np.nan)和异常值(3000)
    'Area':  [50, 70, 80, 80, 100, 70, 120, 150, 90, 110], # 第2行和第5行数据完全重复
    'Age':   [5, 10, 12, 10, 20, 10, 25, 30, 15, 18],      # 第2行和第5行数据完全重复
    'Signal':[1.1, 1.2, 1.1, 9.9, 1.3, 1.2, 1.4, 1.3, 1.5, 1.4] # 包含明显的噪声尖峰 (9.9)
}

df = pd.DataFrame(data)
# 人为制造完全重复的行 (复制 index=1 的行到末尾)
df = pd.concat([df, df.iloc[[1]]], ignore_index=True)

print("原始数据集:")
display(df)

原始数据集:


,Price,Area,Age,Signal
0,100.0,50,5,1.1
1,150.0,70,10,1.2
2,NaN,80,12,1.1
3,200.0,80,10,9.9
4,250.0,100,20,1.3
5,150.0,70,10,1.2
6,300.0,120,25,1.4
7,3000.0,150,30,1.3
8,180.0,90,15,1.5
9,210.0,110,18,1.4


## 数据清洗

In [22]:
df_clean = df.copy()

# ==========================================
# 1.1 缺失值处理: 将 'Price' 列的缺失值填充为该列的中位数
# ==========================================
# TODO: 计算 'Price' 列的中位数并填充 np.nan
# print(len(df['Price'])//2)
# print(sorted(df['Price']))
price_median = sorted(df['Price'])[len(df['Price'])//2]
print(price_median)
df_clean['Price'] =  df_clean['Price'].fillna(value=price_median)

# ==========================================
# 1.2 重复值处理: 删除 df_clean 中的完全重复行，保留第一次出现的行
# ==========================================
# TODO: 删除重复行，并重置索引 (drop=True)
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)

# ==========================================
# 1.3 噪声平滑: 使用滑动平均(Moving Average)平滑 'Signal' 列
# ==========================================
# TODO: 使用窗口大小为 3，居中对齐 (center=True)，最小周期为 1 的滑动平均平滑 'Signal' 列
smoothed_signal = df_clean['Signal'].rolling(window=3, center=True, min_periods=1).mean()
df_clean['Signal'] = smoothed_signal

# # ==========================================
# # 1.4 异常值检测: 使用 IQR (四分位距) 法检测并剔除 'Price' 列的异常值
# # ==========================================
# # TODO: 计算 Q1 (25%) 和 Q3 (75%)，计算 IQR，保留 [Q1 - 1.5*IQR, Q3 + 1.5*IQR] 范围内的行
# Q1 = df_clean['Price'].quantile(0.25)
# Q3 = df_clean['Price'].quantile(0.75)
# IQR = Q3 - Q1
# df_clean = df_clean[(df_clean['Price'] >= Q1 - 1.5 * IQR) & (df_clean['Price'] <= Q3 + 1.5 * IQR)]


# --- 自动判题区 (请勿修改) ---
expected_1_1 = [100., 150., 180., 200., 250., 150., 300., 3000., 180., 210., 150.]
expected_1_2_len = 10
expected_1_3 = [1.15, 1.13333333, 4.06666667, 4.1, 4.13333333, 1.3, 1.3, 1.4, 1.4, 1.45]
expected_1_4_len = 9

print("\n--- 1. 数据清洗 自动判题 ---")
# 独立验证各个步骤
# (略微简化判题：假设你是按顺序就地修改的)
try:
    check_result("1.1 缺失值", df_clean.loc[2, 'Price'] if 2 in df_clean.index else None, [180.0])
    check_result("1.2 重复值", len(df_clean) if df_clean is not None else None, [9]) # 最后应该剩9行 (去重去异常后)
    check_result("1.3 噪声平滑", df_clean['Signal'].iloc[3] if len(df_clean)>3 else None, [1.533333]) # 粗略检查
except:
    print("请先完成上方 TODO 代码块。")
df_clean

180.0

--- 1. 数据清洗 自动判题 ---
✅ [1.1 缺失值] 测试通过! (相对误差: 0.0000e+00)
✅ [1.2 重复值] 测试通过! (相对误差: 0.0000e+00)
❌ [1.3 噪声平滑] 测试失败! 相对误差过大: 4.5562e-01
❌ [1.3 噪声平滑] 判题时发生代码异常: too many indices for array: array is 0-dimensional, but 1 were indexed


,Price,Area,Age,Signal
0,100.0,50,5,1.150000
1,150.0,70,10,1.133333
2,180.0,80,12,4.066667
3,200.0,80,10,4.100000
4,250.0,100,20,4.200000
5,300.0,120,25,1.333333
6,3000.0,150,30,1.400000
7,180.0,90,15,1.400000
8,210.0,110,18,1.450000


## 数据标准化

In [ ]:
X = np.array([[10.0], [20.0], [30.0], [40.0], [50.0]]) # 测试用特征列

# ==========================================
# 2.1 Min-Max 归一化 (缩放到 0 - 1)
# ==========================================
# TODO: 手动实现或使用 sklearn 的 MinMaxScaler
X_minmax = None 


# ==========================================
# 2.2 Z-Score 标准化 (均值为 0，标准差为 1)
# ==========================================
# TODO: 手动实现或使用 sklearn 的 StandardScaler
X_zscore = None


# ==========================================
# 2.3 MaxAbsScaler (按绝对值最大值缩放)
# ==========================================
# TODO: 将数据除以绝对值的最大值
X_maxabs = None


# ==========================================
# 2.4 Log / Logistic 归一化
# ==========================================
# TODO: Log 归一化 (使用 np.log1p 以处理包含 0 的情况)
X_log = None

# TODO: Logistic 归一化 (Sigmoid 函数)
X_logistic = None


# ==========================================
# 2.5 向量单位化 (L2-normalize)
# ==========================================
# TODO: 对 X 进行 L2 范数归一化 (按列求范数，这里设为整体除以 L2 范数)
# 提示: 可以使用 np.linalg.norm 或 sklearn 的 normalize 函数
X_l2 = None


# --- 自动判题区 (请勿修改) ---
exp_minmax = np.array([[0.], [0.25], [0.5], [0.75], [1.]])
exp_zscore = np.array([[-1.41421356], [-0.70710678], [0.], [0.70710678], [1.41421356]])
exp_maxabs = np.array([[0.2], [0.4], [0.6], [0.8], [1.]])
exp_log    = np.array([[2.39789527], [3.04452244], [3.4339872 ], [3.71357207], [3.93182563]])
exp_logist = np.array([[0.9999546 ], [1.], [1.], [1.], [1.]])
exp_l2     = np.array([[0.13483997], [0.26967994], [0.40451992], [0.53935989], [0.67419986]]) # 默认按列 normalize 的结果

print("\n--- 2. 标准化/归一化 自动判题 ---")
check_result("2.1 Min-Max 归一化", X_minmax, exp_minmax)
check_result("2.2 Z-Score 标准化", X_zscore, exp_zscore)
check_result("2.3 MaxAbsScaler", X_maxabs, exp_maxabs)
check_result("2.4 Log 归一化", X_log, exp_log)
check_result("2.5 Logistic 归一化", X_logistic, exp_logist)
check_result("2.6 L2 向量单位化", X_l2, exp_l2)

In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

def set_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        hidden_dim = 256
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.feature_layers = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU())
        self.output_layer = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.input_layer(x)
        x = self.feature_layers(x)
        x = self.output_layer(x)
        return x
        
    


def load_data(npz_path: str) -> TensorDataset:
    data = np.load(npz_path)
    X = torch.from_numpy(data['X'])
    y = torch.from_numpy(data['y'])
    return TensorDataset(X, y)

def train(model: nn.Module, dataloader: DataLoader, optimizer: optim.Optimizer, criterion: nn.Module, accumulation_steps: int, device: torch.device) -> None:

    for step, (X, y) in enumerate(dataloader):
        # import ipdb; ipdb.set_trace()
        model = model.to(device)
        X = X.to(device)
        y = y.to(device)
        out = model(X)
        loss = criterion(out, y)
        loss = loss / accumulation_steps
        loss.backward()

        if (step+1) % accumulation_steps == 0 or (step+1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()





if __name__ == '__main__':
    set_seed()
    npz_path = 'classification_data.npz'
    dataset = load_data(npz_path)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

    input_dim = dataset.tensors[0].shape[1]
    num_classes = len(torch.unique(dataset.tensors[1]))

    model = Classifier(input_dim=input_dim, num_classes=num_classes)
    optimizer = optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    accumulation_steps = 4
    for epoch in range(30):
        train(model, dataloader, optimizer, criterion, accumulation_steps, device)